# Visualize motion path in recording playback

<video controls src="./assets/path_seeking_recording.webm">


Load the nanotube + methane example recording as an MDAnalsis universe:

In [1]:
from nanover.mdanalysis import universe_from_recording

universe = universe_from_recording("../recordings/nanotube-example-recording.nanover.zip")

Find the centroid of the methane molecule for each frame:

In [2]:
import numpy as np

NANOTUBE_ATOMS = list(range(0, 60))
METHANE_ATOMS = list(range(60, 65))

METHANE_CENTROIDS = []
for i, _ in enumerate(universe.trajectory):
    positions = universe.atoms.positions[METHANE_ATOMS] / 10  # angstrom -> nm
    centroid = np.mean(positions, axis=0)
    METHANE_CENTROIDS.append(centroid)

Set up the server with playback of the universe:

In [3]:
from nanover.app import OmniRunner
from nanover.mdanalysis import UniverseSimulation

simulation = UniverseSimulation.from_universe(universe, name="nanotube + methane")
simulation.playback_factor = 30
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: trajectory seeker")
imd_runner.load(0)

C:\Users\ragzo\anaconda3\envs\nanover-dev\Lib\site-packages\MDAnalysis\coordinates\base.py:730: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt


Import the jupyter utilities for drawing + interaction:

In [4]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

Add a line to the scene that shows the path of the centroid over time:

In [5]:
utilities.objects.update_line("path", positions=METHANE_CENTROIDS, size=0.01)

Define a mode that shows the nearest point on the centroid path as the user controllers hover close to it, and seeks the recording when they click:

In [6]:
from nanover.jupyter import Mode
from intersection import closest_point_on_polyline


class SeekLineMode(Mode):
    def on_button_pressed(self, *, key: str, cursor: dict, button: str):
        if button == "primary":
            next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
            result = closest_point_on_polyline(METHANE_CENTROIDS, next_pos)
            utilities.notify_all(f"seek to frame #{result.index}")
            simulation.seek_to_frame_index(result.index)

    def on_cursor_updated(self, *, key: str, cursor: dict):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        result = closest_point_on_polyline(METHANE_CENTROIDS, next_pos)

        if result.distance < 0.05:
            utilities.objects.update_shape(f"cursor.{key}", position=result.point, size=0.02)
        else:
            utilities.objects.remove_shape(f"cursor.{key}")


utilities.use_interaction_modes()
utilities.add_interaction_mode(SeekLineMode, "seek", icon="⏩")

C:\Users\ragzo\anaconda3\envs\nanover-dev\Lib\site-packages\MDAnalysis\coordinates\base.py:730: UserWarning: Reader has no dt information, set to 1.0 ps
  return self.ts.dt
